In [16]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score
)





print("Ready")


Ready


In [17]:
DATASETS = {
    "ds1":  "data/S07-hw-dataset-01.csv",
    "ds2":  "data/S07-hw-dataset-02.csv",
    "ds3":  "data/S07-hw-dataset-03.csv",
}

dfs = {k: pd.read_csv(v) for k, v in DATASETS.items()}


In [18]:
def collect_eda(df):
    return {
        "samples": len(df),
        "features": df.shape[1] - 1,
        "missing": int(df.isna().sum().sum()),
        "dtypes": df.dtypes.astype(str).to_dict()
    }

eda_summary = {k: collect_eda(df) for k, df in dfs.items()}


In [19]:
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


In [20]:
def compute_metrics(X, labels):
    return {
        "silhouette": float(silhouette_score(X, labels)),
        "davies_bouldin": float(davies_bouldin_score(X, labels)),
        "calinski_harabasz": float(calinski_harabasz_score(X, labels))
    }


In [21]:
metrics_summary = {}
best_configs = {}
labels_store = {}

for name, df in dfs.items():
    X = df.drop(columns=["sample_id"])
    Xp = preprocessor.fit_transform(X)

    sil_scores = []
    ks = range(2, 11)

    for k in ks:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(Xp)
        sil_scores.append(silhouette_score(Xp, labels))

    best_k = ks[np.argmax(sil_scores)]

    plt.figure()
    plt.plot(ks, sil_scores, marker="o")
    plt.xlabel("k")
    plt.ylabel("Silhouette")
    plt.title(f"{name}: silhouette vs k")
    plt.savefig(f"artifacts/figures/{name}_silhouette_vs_k.png")
    plt.close()

    km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    labels = km.fit_predict(Xp)

    metrics_summary.setdefault(name, {})["KMeans"] = compute_metrics(Xp, labels)
    best_configs[name] = {"method": "KMeans", "k": best_k}

    labels_store[name] = pd.DataFrame({
        "sample_id": df["sample_id"],
        "cluster_label": labels
    })


In [22]:
for name, df in dfs.items():
    X = df.drop(columns=["sample_id"])
    Xp = preprocessor.fit_transform(X)

    db = DBSCAN(eps=0.5, min_samples=5)
    labels = db.fit_predict(Xp)

    noise_ratio = (labels == -1).mean()

    mask = labels != -1
    if mask.sum() > 10:
        m = compute_metrics(Xp[mask], labels[mask])
        m["noise_ratio"] = float(noise_ratio)
        metrics_summary[name]["DBSCAN"] = m


In [23]:
for name, df in dfs.items():
    X = df.drop(columns=["sample_id"])
    Xp = preprocessor.fit_transform(X)

    agg = AgglomerativeClustering(n_clusters=best_configs[name]["k"], linkage="ward")
    labels = agg.fit_predict(Xp)

    metrics_summary[name]["Agglomerative"] = compute_metrics(Xp, labels)


In [24]:
for name, df in dfs.items():
    X = df.drop(columns=["sample_id"])
    Xp = preprocessor.fit_transform(X)

    labels = labels_store[name]["cluster_label"]
    pca = PCA(n_components=2, random_state=42)
    X2 = pca.fit_transform(Xp)

    plt.figure()
    plt.scatter(X2[:,0], X2[:,1], c=labels, s=10)
    plt.title(f"{name}: PCA(2D)")
    plt.savefig(f"artifacts/figures/{name}_pca.png")
    plt.close()


In [25]:
X = dfs["ds1"].drop(columns=["sample_id"])
Xp = preprocessor.fit_transform(X)

labels_list = []
for rs in range(5):
    km = KMeans(n_clusters=best_configs["ds1"]["k"], random_state=rs, n_init=10)
    labels_list.append(km.fit_predict(Xp))

ari_scores = [
    adjusted_rand_score(labels_list[0], labels_list[i])
    for i in range(1, 5)
]

stability_mean_ari = float(np.mean(ari_scores))


In [26]:
with open("artifacts/metrics_summary.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

with open("artifacts/best_configs.json", "w") as f:
    json.dump(best_configs, f, indent=2)

for name, df_lab in labels_store.items():
    df_lab.to_csv( f"artifacts/labels/labels_hw07_{name}.csv",
        index=False
    )


In [29]:
def generate_report():
    lines = []
    lines.append("# HW07: Кластеризация и обучение без учителя\n")

    lines.append("## 1. Описание датасетов\n")
    for k, v in eda_summary.items():
        lines.append(f"### Датасет {k}")
        lines.append(f"- Количество объектов: {v['samples']}")
        lines.append(f"- Количество признаков: {v['features']}")
        lines.append(f"- Общее число пропусков: {v['missing']}\n")

    lines.append("## 2. Использованные модели и метрики качества\n")
    for ds, models in metrics_summary.items():
        lines.append(f"### Датасет {ds}")
        for m, vals in models.items():
            lines.append(
                f"- **{m}**: "
                f"silhouette = {vals['silhouette']:.3f}, "
                f"davies_bouldin = {vals['davies_bouldin']:.3f}, "
                f"calinski_harabasz = {vals['calinski_harabasz']:.1f}"
            )
        lines.append("")

    lines.append("## 3. Проверка устойчивости кластеризации\n")
    lines.append(
        f"Для датасета ds1 была проведена проверка устойчивости алгоритма KMeans. "
        f"Модель запускалась 5 раз с различными значениями random_state. "
        f"Сходство полученных разбиений оценивалось с помощью Adjusted Rand Index (ARI). "
        f"Среднее значение ARI составило {stability_mean_ari:.3f}, "
        f"что свидетельствует о высокой воспроизводимости результата."
    )

    lines.append("\n## 4. Итоговые выводы\n")
    lines.append(
        "В ходе работы были рассмотрены несколько алгоритмов кластеризации без учителя, "
        "включая KMeans, DBSCAN и иерархическую агломеративную кластеризацию. "
        "Алгоритм KMeans показал наилучшие результаты на датасетах с компактными и "
        "приближенно сферическими кластерами. "
        "DBSCAN оказался эффективным для выявления выбросов и шумовых объектов. "
        "Агломеративная кластеризация продемонстрировала сопоставимое качество и "
        "предоставила интерпретируемую иерархическую структуру кластеров. "
        "Выбор наилучших конфигураций осуществлялся на основе внутренних метрик качества "
        "и подтверждался визуализацией данных в пространстве главных компонент (PCA)."
    )

    return "\n".join(lines)


with open("report.md", "w", encoding="utf-8") as f:
    f.write(generate_report())

print("report.md сгенерирован")


report.md сгенерирован
